# Amelie — Models, Search Engine y Visualización

Notebook individual para probar `models.py`, `search_engine.py` y
`visualization.py` contra el dataset real, **sin tocar `main.py`**.

Tus partes dependen de `preprocessing.py` y `tfidf.py` (los de Martín),
así que los importas tal cual están en `src/` — no necesitas modificarlos,
solo usarlos como insumo.

## 1. Clonar el repositorio

In [ ]:
!git clone -b RAMA DONDE SUBIERON SU NOTEBOOK https://github.com/Fifthtaschenmesser4/Taller02-ProgCien.git

Si ya lo clonaste antes y solo quieres traer los cambios más recientes de
tus compañeros, usa esto en vez del clone:

```python
%cd /content/Taller02-ProgCien
!git pull
```

## 2. Instalar dependencias

In [ ]:
!pip install -q -r /content/Taller02-ProgCien/requirements.txt

## 3. Dataset

Asegúrate de que los 3 CSV (`t_asv.csv`, `key_english.csv`,
`key_genre_english.csv`) estén en `data/` — si están en el repo, ya
llegaron con el `git clone`/`git pull`.

In [ ]:
from pathlib import Path
from src.data_loader import cargar_dataset

dir_dataset = Path("data")
df_raw = cargar_dataset(
    dir_dataset / "t_asv.csv",
    dir_dataset / "key_english.csv",
    dir_dataset / "key_genre_english.csv",
)
df_raw.head()

## 4. `models.py` — construir la jerarquía Biblia → Testamento → Libro → Capítulo → Versículo

In [ ]:
from src.models import Biblia

biblia = Biblia.from_dataframe(
    df_raw,
    col_libro="Book Name",
    col_testamento="Testament (OT or NT)",
    col_capitulo="Chapter",
    col_versiculo="Verse",
    col_texto="Text",
    col_genero="Genre name",
)
print(biblia)
print(biblia.get_resumen())
print(biblia.get_resumen_generos())

In [ ]:
# Pruebas puntuales de la jerarquía: navegar un libro y un capítulo
libro = biblia.get_libros()[0]
print(libro)
print("Capítulos:", libro.get_cantidad_capitulos())

capitulo = list(libro.capitulos.values())[0]
print(capitulo)
print(capitulo.get_texto_completo()[:200])

versiculo = capitulo.versiculos[0]
print(versiculo)

In [ ]:
df = biblia.to_dataframe()
df.head()

## 5. Preprocesamiento y TF-IDF (de Martín)

Necesitas esto como insumo para `search_engine.py`. No tienes que tocar
estos archivos, solo usarlos — si algo no calza con lo que esperas
(ej. nombres de columnas), avísale a Martín en vez de modificarlo tú
misma, para evitar choques.

In [ ]:
from src.preprocessing import TextPreprocessor
from src.tfidf import TFIDFVectorizer, cosine_similarity_matrix

preprocessor = TextPreprocessor()
df["texto_procesado"] = preprocessor.process_corpus(df["texto_original"].tolist())

vectorizer = TFIDFVectorizer()
matriz_tfidf_versiculos = vectorizer.fit_transform(df["texto_procesado"].tolist())
matriz_tfidf_versiculos.shape

## 6. `search_engine.py` — motor de búsqueda semántico

In [ ]:
from src.search_engine import SemanticSearchEngine

motor = SemanticSearchEngine(TextPreprocessor(), TFIDFVectorizer())
motor.fit(df)

motor.buscar("amor y fe", k=5)

In [ ]:
# Buscar versículos similares a uno que ya está en el corpus (por índice)
motor.buscar_por_indice(idx_versiculo=0, k=5)

## 7. `visualization.py` — gráficos del corpus

In [ ]:
from src import visualization as viz

viz.plot_longitud_versiculos(df)

In [ ]:
viz.plot_versiculos_por_libro(df)

In [ ]:
# Heatmap de similitud entre libros (OBLIGATORIO según el enunciado)
textos_por_libro = df.groupby("libro")["texto_procesado"].sum()  # concatena listas de tokens

vectorizer_libros = TFIDFVectorizer()
matriz_tfidf_libros = vectorizer_libros.fit_transform(textos_por_libro.tolist())
matriz_similitud_libros = cosine_similarity_matrix(matriz_tfidf_libros)

viz.plot_heatmap_similitud_libros(matriz_similitud_libros, textos_por_libro.index.tolist())

In [ ]:
# PCA de versículos coloreados por testamento y por libro
viz.plot_pca_versiculos(matriz_tfidf_versiculos, df["testamento"], titulo="Versículos por testamento")

In [ ]:
viz.plot_pca_versiculos(matriz_tfidf_versiculos, df["libro"], titulo="Versículos por libro")

In [ ]:
# Bonus: PCA coloreado por género literario (dato nuevo que agregamos en models.py)
viz.plot_pca_versiculos(matriz_tfidf_versiculos, df["genero"], titulo="Versículos por género literario")

## 8. Notas para subir tus cambios

Si modificaste `models.py`, `search_engine.py` o `visualization.py`, desde
Colab puedes hacer commit y push directo:

```python
!git config --global user.email "tu_correo@ucn.cl"
!git config --global user.name "Amelie"
!git add src/models.py src/search_engine.py src/visualization.py notebooks/amelie.ipynb
!git commit -m "Pruebas de models, search_engine y visualization con dataset real"
!git push
```

Necesitas un Personal Access Token de GitHub para el `push` (no tu
contraseña normal). Si prefieres algo más simple, descarga el notebook
(`Archivo > Descargar > .ipynb`) y subilo a mano desde la web de GitHub.

Martín es quien integra todo en `main.py` al final, así que no necesitas
tocar ese archivo.